# SUBLIME — Kaggle Reproduction
**Paper:** [Towards Unsupervised Deep Graph Structure Learning (WWW 2022)](https://arxiv.org/pdf/2201.06367)

This notebook clones the repo, installs dependencies, and runs the SUBLIME experiments.
Use **T4 x2** or **P100** accelerator in Kaggle settings.

## 1. Install dependencies

In [ ]:
import subprocess, sys

# Check CUDA version to install the right DGL build
result = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(result.stdout or result.stderr)

import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('GPU count:', torch.cuda.device_count())

In [ ]:
import subprocess, sys, os

ENV  = '/tmp/dgl071'          # micromamba prefix
PY   = f'{ENV}/bin/python'
MM   = '/tmp/micromamba'      # single-binary micromamba

def run(cmd, check=False):
    print('>>>', cmd[:120])
    try:
        r = subprocess.run(cmd, shell=True, text=True,
                           stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           timeout=600)
    except subprocess.TimeoutExpired:
        print('  [TIMEOUT]'); return 1
    if r.stdout: print(r.stdout[-3000:])
    if check and r.returncode != 0:
        raise RuntimeError(f'Command failed (rc={r.returncode})')
    return r.returncode

# ── 1. micromamba binary ──────────────────────────────────────────────────────
# micromamba is a single static binary — no conda/mamba install needed.
if not os.path.exists(MM):
    run('curl -fsSL https://micro.mamba.pm/api/micromamba/linux-64/latest '
        '| tar -xvj --strip-components=1 -C /tmp bin/micromamba', check=True)
    run(f'chmod +x {MM}')

# ── 2. Create conda environment: Python 3.8 + PyTorch 1.7.1 + CUDA 11.0 ──────
# This is the exact paper environment. pytorch conda channel bundles cudatoolkit.
if not os.path.exists(PY):
    run(f'{MM} create -p {ENV} -y '
        f'python=3.8 pytorch=1.7.1 cudatoolkit=11.0 '
        f'-c pytorch -c nvidia -c conda-forge -c defaults --no-rc', check=True)

# ── 3. DGL 0.7.1 (CUDA 11.0 build) from the dglteam conda channel ────────────
# dgl-cuda11.0=0.7.1 is the conda package for DGL 0.7.1 compiled against cu110.
# It was never released as a pip wheel — micromamba is the only way to get it.
run(f'{MM} install -p {ENV} -y '
    f'dgl-cuda11.0=0.7.1 -c dglteam -c conda-forge -c defaults --no-rc', check=True)

# ── 4. Exact paper Python deps via pip ────────────────────────────────────────
run(f'{PY} -m pip install -q '
    f'numpy==1.20.2 scipy==1.6.3 scikit-learn==0.24.2 munkres==1.1.4 networkx',
    check=True)

# ── 5. Verify ─────────────────────────────────────────────────────────────────
run(f"{PY} -c 'import torch, dgl; print(torch.__version__, dgl.__version__, torch.cuda.is_available())'", check=True)


In [ ]:
# This verifies the Kaggle-default Python 3.12 + DGL 2.1 env.
# (The exact paper env is set up in the cell above and used via subprocess below.)
import sys, types

class _DGLGraphboltStub(types.ModuleType):
    def __getattr__(self, name):
        if name.startswith('__') and name.endswith('__'):
            raise AttributeError(name)
        cls = type(name, (), {})
        setattr(self, name, cls)
        return cls

_gb_stub = _DGLGraphboltStub('dgl.graphbolt')
_gb_stub.__file__ = '<dgl-graphbolt-stub>'
_gb_stub.__path__ = []
_gb_stub.__package__ = 'dgl.graphbolt'
_gb_stub.__spec__ = None
sys.modules['dgl.graphbolt'] = _gb_stub

import dgl
print('Kaggle env — Python:', sys.version.split()[0], '| DGL:', dgl.__version__)


## 2. Clone the repo

In [ ]:
import os

REPO_DIR = '/kaggle/working/SUBLIME'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/wishaalk/SUBLIME.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())
!ls

## 3. Verify data files

In [ ]:
!ls data/

## 4. Quick smoke-test (1 trial, reduced epochs)

This uses **Cora / structure inference** — the first experiment from Table 2 of the paper.  
We set `-ntrials 1` and `-epochs 200` just to confirm everything runs end-to-end before launching the full experiment.

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset cora \
    -ntrials 1 \
    -sparse 0 \
    -epochs_cls 200 \
    -lr_cls 0.001 \
    -w_decay_cls 0.0005 \
    -hidden_dim_cls 32 \
    -dropout_cls 0.5 \
    -dropedge_cls 0.25 \
    -nlayers_cls 2 \
    -patience_cls 10 \
    -epochs 200 \
    -lr 0.01 \
    -w_decay 0.0 \
    -hidden_dim 512 \
    -rep_dim 256 \
    -proj_dim 256 \
    -dropout 0.5 \
    -dropedge_rate 0.5 \
    -nlayers 2 \
    -type_learner fgp \
    -k 30 \
    -sim_function cosine \
    -activation_learner relu \
    -gsl_mode structure_inference \
    -eval_freq 20 \
    -tau 1 \
    -maskfeat_rate_learner 0.5 \
    -maskfeat_rate_anchor 0.7 \
    -contrast_batch_size 0 \
    -c 0 \
    -gpu 0

---
## 5. Full experiments (paper settings)

Run these cells one at a time. Each maps to a row in the paper's tables.

Expected results from the paper (Table 2 / Table 3):

| Dataset | Mode | Task | Reported Acc |
|---------|------|------|--------------|
| Cora | Structure Inference | Classification | 83.6% |
| Cora | Structure Refinement | Classification | 84.7% |
| Cora | Structure Refinement | Clustering (ACC) | 79.8% |
| Citeseer | Structure Inference | Classification | 73.6% |
| Citeseer | Structure Refinement | Classification | 73.9% |
| Citeseer | Structure Refinement | Clustering (ACC) | 69.6% |
| Pubmed | Structure Inference | Classification | 79.9% |
| Pubmed | Structure Refinement | Classification | 81.3% |

### 5a. Cora — Node Classification @ Structure Inference

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset cora -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 \
    -epochs 4000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner fgp -k 30 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_inference -eval_freq 20 -tau 1 \
    -maskfeat_rate_learner 0.5 -maskfeat_rate_anchor 0.7 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5b. Cora — Node Classification @ Structure Refinement

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset cora -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.75 -nlayers_cls 2 -patience_cls 10 \
    -epochs 4000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner fgp -k 30 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_refinement -eval_freq 50 -tau 0.9999 \
    -maskfeat_rate_learner 0.7 -maskfeat_rate_anchor 0.6 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5c. Cora — Node Clustering @ Structure Refinement

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset cora -downstream_task clustering -ntrials 10 -sparse 0 \
    -epochs 2500 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner fgp -k 20 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_refinement -eval_freq 100 -tau 0.9999 \
    -maskfeat_rate_learner 0.1 -maskfeat_rate_anchor 0.8 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5d. Citeseer — Node Classification @ Structure Inference

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset citeseer -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.05 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.5 -nlayers_cls 2 -patience_cls 10 \
    -epochs 1000 -lr 0.01 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 \
    -type_learner att -k 20 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_inference -eval_freq 50 -tau 0.9999 \
    -maskfeat_rate_learner 0.8 -maskfeat_rate_anchor 0.7 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5e. Citeseer — Node Classification @ Structure Refinement

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset citeseer -ntrials 5 -sparse 0 \
    -epochs_cls 200 -lr_cls 0.001 -w_decay_cls 0.05 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.5 -nlayers_cls 2 -patience_cls 10 \
    -epochs 1000 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 \
    -type_learner att -k 20 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_refinement -eval_freq 20 -tau 0.9999 \
    -maskfeat_rate_learner 0.6 -maskfeat_rate_anchor 0.8 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5f. Citeseer — Node Clustering @ Structure Refinement

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset citeseer -downstream_task clustering -ntrials 10 -sparse 0 \
    -epochs 1000 -lr 0.001 -w_decay 0.0 -hidden_dim 512 -rep_dim 256 -proj_dim 256 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner att -k 20 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_refinement -eval_freq 100 -tau 0.999 \
    -maskfeat_rate_learner 0.4 -maskfeat_rate_anchor 0.9 \
    -contrast_batch_size 0 -c 0 -gpu 0

### 5g. Pubmed — Node Classification @ Structure Inference

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset pubmed -ntrials 5 -sparse 1 \
    -epochs_cls 200 -lr_cls 0.01 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 \
    -epochs 2000 -lr 0.01 -w_decay 0.0 -hidden_dim 128 -rep_dim 64 -proj_dim 64 \
    -dropout 0.5 -dropedge_rate 0.25 -nlayers 2 \
    -type_learner att -k 15 -sim_function cosine -activation_learner tanh \
    -gsl_mode structure_inference -eval_freq 20 -tau 1 \
    -maskfeat_rate_learner 0.8 -maskfeat_rate_anchor 0.3 \
    -contrast_batch_size 2000 -c 0 -gpu 0

### 5h. Pubmed — Node Classification @ Structure Refinement

In [ ]:
!/tmp/dgl071/bin/python main.py \
    -dataset pubmed -ntrials 5 -sparse 1 \
    -epochs_cls 200 -lr_cls 0.01 -w_decay_cls 0.0005 -hidden_dim_cls 32 \
    -dropout_cls 0.5 -dropedge_cls 0.25 -nlayers_cls 2 -patience_cls 10 \
    -epochs 1500 -lr 0.001 -w_decay 0.0 -hidden_dim 128 -rep_dim 64 -proj_dim 64 \
    -dropout 0.5 -dropedge_rate 0.5 -nlayers 2 \
    -type_learner mlp -k 10 -sim_function cosine -activation_learner relu \
    -gsl_mode structure_refinement -eval_freq 20 -tau 0.999 \
    -maskfeat_rate_learner 0.4 -maskfeat_rate_anchor 0.4 \
    -contrast_batch_size 2000 -c 50 -gpu 0